#### get text content from data

In [1]:
import pandas as pd
import re
import json
import ast
from os import fdopen, remove
from tempfile import mkstemp
from shutil import move, copymode
from pprint import pprint
import os
import string
from importlib.resources import path
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

def flatten(xss):
    return [x for xs in xss for x in xs]

In [2]:
# preprocessing auxiliary functions

stopwords = open("config_files/spacy_stopwords.txt").read().replace("{", '[').replace("}",']')
stopwords = ast.literal_eval(stopwords)

# this one is not needed if working on Sentence_spacy
def remove_punctuation(corpus):
    cleaned_corpus = corpus.translate(str.maketrans('', '', string.punctuation))
    return(cleaned_corpus)

# remove stop words from text
def remove_stop_words_from_text(cleaned_corpus):
    out = [word for word in cleaned_corpus if not word in stopwords]
    out_text = ' '.join(out)
    return out_text  # , out


def remove_stop_words(token_list):
    return [token for token in token_list if not token.is_stop]

    
def lemmatizing(token_list):
    return [token.lemma_ for token in token_list]
    
def get_POS(token_list):
    return [token.pos_ for token in token_list]

In [3]:
# AI-terms from ANTHROSCORE paper, transalted with deepL
ai_terms = list(ast.literal_eval(open("config_files/ger_anthroscore_KI_terms.txt").read()).values())
regex_ai = "".join(["\\b"+x.replace(" ","\\s")+"\\b|" for x in ai_terms])[:-1]

# get AI-related words, sorted according to 'Häufigkeitsklassen'
# https://corpora.uni-leipzig.de/de/res?corpusId=deu_news_2024&word=KI
ki_terms = ['KI','Künstliche Intelligenz', 'ChatGPT', 'AI', 'Copilot', 'Gemini', 'Chatbot','LLM', 'Chat-GPT',
            'Sprachmodell', 'KI-Modell', 'KI-System', 'KI-Tool', 'Artificial Intelligence', 'KI-Anwendung'] + ai_terms
ki_terms = [x.lower() for x in ki_terms]
regex_ki = "".join(["\\b"+x.replace(" ","\\s")+"\\b|" for x in ki_terms])[:-1]
print(regex_ki)

\bki\b|\bkünstliche\sintelligenz\b|\bchatgpt\b|\bai\b|\bcopilot\b|\bgemini\b|\bchatbot\b|\bllm\b|\bchat-gpt\b|\bsprachmodell\b|\bki-modell\b|\bki-system\b|\bki-tool\b|\bartificial\sintelligence\b|\bki-anwendung\b|\bpalm\b|\blms\b|\bllama\b|\btransformer\b|\bsprachmodelle\b|\bsprachmodell\b|\bgpt\b|\bplms\b|\bvortrainierte\ssprachmodelle\b|\bgpt-2\b|\bxlnet\b|\bgroße\ssprachmodelle\b|\bllms\b|\bgpt-3\b|\bstiftungsmodell\b|\bgpt-neo\b|\bgpt-j\b|\bchatgpt\b|\bgpt-4\b


In [4]:
# get sentences with ki_terms
def to_sentences(sentences_file):
    sentences = []
    with open(sentences_file) as f:
        for line in f:
            txt = line.split('\t')
            #split sentences
            txt_content = txt[1::]
            # filter ki_terms
            sentences.append(remove_punctuation(line.lower()))  
    return sentences


leipzig_main = "deu_news"
ki_data = {}
sentences_all = {}
for p in os.listdir(leipzig_main):
    k = p.replace("deu_news_","")
    sentences_file = os.path.join(leipzig_main, p, f"{p}-sentences.txt")
    # extract ki_sentences
    ki_sentences = {}
    sentences_all[k] = to_sentences(sentences_file)
    for sent in sentences_all[k]:       
        if len(sent) > 0 and re.search(regex_ki, sent):
            #print("res",sent, re.search(regex_ki, sent).string)
            split = re.sub(r"\t", "-", sent).split("-")
            ki_sentences[split[0]] = split[1]
    #print(ki_sentences)
    print(f"{k}:", len(ki_sentences))
    ki_data[k] = ki_sentences

with open(r'generated_data/Leipzig_KI_sentences_all.json', 'w') as f:
    json.dump(ki_data, f)

# save ki_sentences to file
with open(r"generated_data/Leipzig_KI_sentences_all.txt", "w") as f:
        f.write(str(ki_data))

2003_1M: 104
2013_1M: 213
2007_1M: 278
1996_1M: 124
2017_1M: 418
2023_1M: 2414
2006_1M: 195
1997_1M: 118
2016_1M: 361
2022_1M: 422
2002_1M: 153
2012_1M: 425
2009_1M: 404
2019_1M: 636
1998_1M: 95
2024_10K: 35
2008_1M: 266
2018_1M: 503
1999_1M: 159
2005_1M: 143
2015_1M: 361
2021_1M: 420
2001_1M: 175
2025_1M: 3357
2011_1M: 508
2000_1M: 235
2024_1M: 2575
2010_1M: 398
2004_1M: 113
1995_1M: 116
2014_1M: 300
2020_1M: 403


In [5]:
sent_dict = {}
for (key, sentences) in (sentences_all.items()):
    for sent in sentences:
        split = re.sub(r"\t", "-", sent).split("-")
        if (len(split) > 1):
            sent_dict[f"{key}_{split[0]}"] = split[1]


In [6]:
# get KI-terms in "...words.txt" file
words_all = {}
for p in os.listdir(leipzig_main):
    k = p.replace("deu_news_","")
    words_file = os.path.join(leipzig_main, p, f"{p}-words.txt")
    # Convert words file to tab-separated for csv reading
    #Create temp file
    fh, abs_path = mkstemp()
    with fdopen(fh,'w') as new_file:
        with open(words_file) as old_file:
            for line in old_file:
                new_file.write(line.replace(' ',';').replace('\t',';').replace('"','\\"'))
    #Copy the file permissions from the old file to the new file, then remove the old file and
    copymode(words_file, abs_path)
    remove(words_file)
    move(abs_path, words_file)

    # filter ki_terms
    data = pd.read_csv(words_file, sep=";", encoding='utf-8', header=None, on_bad_lines='skip', escapechar='\\')
    words_all[k] = data[1].tolist()

ki_words = list(set([w.lower() for sub in words_all.values() for w in sub if isinstance(w, str) and w.lower() in ki_terms]))
print("percentage of ki_words in ki_terms:", len(ki_words)/len(ki_terms))
pprint(ki_words)


/var/folders/0h/lr036jk55nq7b5pmf0kg0qq40000gn/T/ipykernel_16613/3607378963.py:19: DtypeWarning: Columns (2) have mixed types. Specify dtype option on import or set low_memory=False.
  data = pd.read_csv(words_file, sep=";", encoding='utf-8', header=None, on_bad_lines='skip', escapechar='\\')
/var/folders/0h/lr036jk55nq7b5pmf0kg0qq40000gn/T/ipykernel_16613/3607378963.py:19: DtypeWarning: Columns (2) have mixed types. Specify dtype option on import or set low_memory=False.
  data = pd.read_csv(words_file, sep=";", encoding='utf-8', header=None, on_bad_lines='skip', escapechar='\\')
/var/folders/0h/lr036jk55nq7b5pmf0kg0qq40000gn/T/ipykernel_16613/3607378963.py:19: DtypeWarning: Columns (2) have mixed types. Specify dtype option on import or set low_memory=False.
  data = pd.read_csv(words_file, sep=";", encoding='utf-8', header=None, on_bad_lines='skip', escapechar='\\')


percentage of ki_words in ki_terms: 0.7647058823529411
['gemini',
 'lms',
 'ki-system',
 'gpt-3',
 'gpt-4',
 'sprachmodelle',
 'sprachmodell',
 'chatbot',
 'ki',
 'llama',
 'palm',
 'ki-tool',
 'gpt-j',
 'chatgpt',
 'ki-anwendung',
 'transformer',
 'llm',
 'plms',
 'llms',
 'ki-modell',
 'gpt',
 'stiftungsmodell',
 'copilot',
 'gpt-2',
 'ai',
 'chat-gpt']


### TF-IDF

In [7]:
# use tf-idf to get keywords from texts
# corpus a list of sentences/documents
# stopwords_l a list of stopwords in the target language if not english
def calc_tf_idf(corpus, stopwords_l= None):
    corpus_cleaned = [remove_punctuation(doc.lower()) for doc in corpus]
    if stopwords_l:
        corpus_cleaned = [remove_stop_words_from_text(doc.split()) for doc in corpus_cleaned]
        vectorizer = TfidfVectorizer(stop_words=stopwords_l, lowercase=True, ngram_range=(1,2))
    else:
        vectorizer = TfidfVectorizer(stop_words='english', lowercase=True, ngram_range=(1,2))
    # fit tfidf
    tfidf_m = vectorizer.fit_transform(corpus_cleaned) #transform
    #print(tfidf_m.shape)
    feature_array = vectorizer.get_feature_names_out()
    # top (20?) for every doc
    tfidf_sorting = np.argsort(tfidf_m.toarray()).flatten()[::-1] 
    top_n = feature_array[tfidf_sorting].tolist()[:20]
    print(top_n)
    return top_n #array of list(s?)

In [8]:
tfidf_words = {}
for k in ki_data.keys():
    tfidf_words[k] = calc_tf_idf(ki_data[k].values(), stopwords)

with open(f"generated_data/Leipzig_KI_sentences_tfidf.txt", "w") as f:
    json.dump(tfidf_words, f)


['spielerin sprung', 'viertelfinale', 'spielerin', 'sprung', 'sprung viertelfinale', '75 63', 'williams usa', 'williams', 'japan satzverlust', 'titelverteidigerin', 'titelverteidigerin serena', '63', 'usa 75', 'serena williams', 'serena', '63 ai', 'schaffte spielerin', 'schaffte', 'satzverlust schaffte', 'satzverlust']
['partnern cap', 'gemini deloitte', 'barber sas', 'barber', 'verantwortlich', 'neuer vertriebswege', 'zuvor nick', 'sas institute', 'sas', 'cap gemini', 'cap', 'aufbau neuer', 'partnern', 'grundlegenden aufbau', 'institute', 'institute grundlegenden', 'grundlegenden', 'vertriebswege', 'vertriebswege partnern', 'deloitte accenture']
['rechts', 'öffentlichen', 'öffentlichen rechts', 'gmbh anstalt', 'janssen erscheint', 'rechts janssen', 'rechts stiftung', 'prüfung formen', 'prüfung', 'stünde prüfung', 'stiftung', 'stiftung öffentlichen', 'stiftungsmodell wahrscheinlich', 'wahrscheinlich', 'wahrscheinlich gängigste', 'gängigste', 'klassische gmbh', 'klassische', 'erscheint 

In [9]:
# TODO: tf-idf means per category?? which categories, ai terms?
''' 
vectorizer = TfidfVectorizer(lowercase= True)
means = {}
for i,doc in enumerate(corpora_lex.values()):
    if doc:
        X = vectorizer.fit_transform(doc).todense()
        # calc means for docs
        mean = np.mean(X)
        # to dict
        means[list(corpora_lex.keys())[i]] = mean
        mean_sorted = sorted(means.items(), key = lambda x : x[1], reverse = True)
pprint(mean_sorted)
'''

' \nvectorizer = TfidfVectorizer(lowercase= True)\nmeans = {}\nfor i,doc in enumerate(corpora_lex.values()):\n    if doc:\n        X = vectorizer.fit_transform(doc).todense()\n        # calc means for docs\n        mean = np.mean(X)\n        # to dict\n        means[list(corpora_lex.keys())[i]] = mean\n        mean_sorted = sorted(means.items(), key = lambda x : x[1], reverse = True)\npprint(mean_sorted)\n'